# EdVyro — Customer Churn Data Quality & Exploratory Analysis

**Task:** Data Quality and Exploratory Analysis (Task 02)

**Decision question:** Which customer groups should a fictional subscription team contact first to reduce avoidable churn?

This notebook is designed to satisfy the EdVyro passing checklist:
1. Profile columns, data types, missing values, outliers, and duplicates.
2. Clean the data with documented assumptions and validation checks.
3. Calculate summary statistics and inspect meaningful distributions.
4. Export the cleaned CSV, quality summary, supporting tables, charts, and an Excel workbook.

**Important:** Only the supplied EdVyro dataset is used. No alternative churn dataset is substituted.

In [ ]:
# 1. Setup
import io
import os
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

SOURCE_URL = "https://edvyro.in/datasets/customer_churn_sample.csv"
DATA_DIR = Path("customer_churn_outputs")
DATA_DIR.mkdir(exist_ok=True)
RAW_PATH = DATA_DIR / "customer_churn_raw.csv"
CLEAN_PATH = DATA_DIR / "customer_churn_cleaned.csv"
EXCEL_PATH = DATA_DIR / "customer_churn_data_quality_report.xlsx"
REPORT_PATH = DATA_DIR / "submission_report.md"

EXPECTED = [
    "customer_id", "tenure_months", "monthly_charge",
    "support_tickets", "contract_type", "churned"
]
ALLOWED_CONTRACTS = {"monthly", "quarterly", "annual"}

print("Output folder:", DATA_DIR.resolve())

In [ ]:
# 2. Load the supplied EdVyro dataset
# First try the official dataset URL. If the environment blocks direct CSV downloads,
# a file-upload fallback is provided so the analysis can still be completed with the
# exact supplied CSV.
try:
    urllib.request.urlretrieve(SOURCE_URL, RAW_PATH)
    print("Downloaded supplied dataset from EdVyro.")
except Exception as download_error:
    print("Direct download was unavailable:", type(download_error).__name__)
    print("Please upload the supplied customer_churn_sample.csv when prompted.")
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_candidates = [name for name in uploaded if name.lower().endswith('.csv')]
        if not csv_candidates:
            raise FileNotFoundError("No CSV file was uploaded.")
        selected = csv_candidates[0]
        RAW_PATH.write_bytes(uploaded[selected])
        print("Uploaded:", selected)
    except ImportError as exc:
        raise RuntimeError(
            "The EdVyro CSV could not be downloaded in this environment. "
            "Run this notebook in Google Colab or place the supplied CSV in the working folder."
        ) from exc

raw = pd.read_csv(RAW_PATH)
print("Raw shape:", raw.shape)
display(raw.head())

In [ ]:
# 3. Standardize field names and types

df = raw.copy()
df.columns = (
    df.columns.astype(str).str.strip().str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

missing_expected = [c for c in EXPECTED if c not in df.columns]
extra_columns = [c for c in df.columns if c not in EXPECTED]
if missing_expected:
    raise ValueError(f"Required EdVyro fields are missing: {missing_expected}")

# Keep the six fields defined by the EdVyro source guide.
df = df[EXPECTED].copy()

df["customer_id"] = df["customer_id"].astype("string").str.strip()
for c in ["tenure_months", "monthly_charge", "support_tickets"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["contract_type"] = df["contract_type"].astype("string").str.strip().str.lower()

# Normalize common binary encodings only; unrecognized values become missing and are audited.
churn_map = {
    "1": 1, "0": 0, "true": 1, "false": 0,
    "yes": 1, "no": 0, "y": 1, "n": 0,
    "churned": 1, "active": 0, "churn": 1, "not_churned": 0,
}
df["churned"] = df["churned"].astype("string").str.strip().str.lower().map(churn_map)
df["churned"] = pd.to_numeric(df["churned"], errors="coerce")

print("Standardized columns:", list(df.columns))
display(df.dtypes.to_frame("dtype"))

In [ ]:
# 4. Initial data-quality profile
profile = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "rows": len(df),
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [round(float(df[c].isna().mean() * 100), 2) for c in df.columns],
    "unique_values": [int(df[c].nunique(dropna=True)) for c in df.columns],
})

duplicate_rows = int(df.duplicated().sum())
duplicate_customer_ids = int(df["customer_id"].duplicated(keep=False).sum())

print("Duplicate full rows:", duplicate_rows)
print("Rows involved in duplicate customer IDs:", duplicate_customer_ids)
display(profile)

In [ ]:
# 5. Audit invalid/impossible values BEFORE cleaning
quality_flags = pd.DataFrame(index=df.index)
quality_flags["missing_required"] = df[EXPECTED].isna().any(axis=1)
quality_flags["exact_duplicate_row"] = df.duplicated(keep=False)
quality_flags["duplicate_customer_id"] = df["customer_id"].duplicated(keep=False)
quality_flags["invalid_tenure"] = df["tenure_months"].notna() & (df["tenure_months"] < 0)
quality_flags["invalid_monthly_charge"] = df["monthly_charge"].notna() & (df["monthly_charge"] < 0)
quality_flags["invalid_support_tickets"] = (
    df["support_tickets"].notna() &
    ((df["support_tickets"] < 0) | (df["support_tickets"] % 1 != 0))
)
quality_flags["invalid_contract_type"] = (
    df["contract_type"].notna() & ~df["contract_type"].isin(ALLOWED_CONTRACTS)
)
quality_flags["invalid_churned"] = df["churned"].notna() & ~df["churned"].isin([0, 1])

flag_counts = quality_flags.sum().astype(int).sort_values(ascending=False).to_frame("rows_flagged")
display(flag_counts)

In [ ]:
# 6. Outlier audit using the IQR rule
numeric_cols = ["tenure_months", "monthly_charge", "support_tickets"]
outlier_rows = []

for col in numeric_cols:
    s = df[col].dropna()
    if s.empty:
        outlier_rows.append([col, np.nan, np.nan, np.nan, 0, 0, "No non-missing values"])
        continue
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[col] < lower) | (df[col] > upper)
    outlier_rows.append([col, q1, q3, iqr, lower, upper, int(mask.sum())])

outlier_report = pd.DataFrame(
    outlier_rows,
    columns=["column", "Q1", "Q3", "IQR", "lower_bound", "upper_bound", "outlier_count"]
)
display(outlier_report)
print("Decision: IQR outliers are flagged for review, not automatically deleted. Extreme is not the same as impossible.")

## Cleaning assumptions and decision log

| Issue | Decision | Rationale |
|---|---|---|
| Column-name variation | Standardize names to the six source-guide fields | Makes the workflow reproducible without changing field meaning |
| Missing required values | Exclude affected rows from the analysis-ready dataset | No unsupported imputation is appropriate for this short audit |
| Exact duplicate rows | Remove duplicate copies, keeping the first occurrence | Duplicate records do not represent additional customers |
| Duplicate customer IDs | After exact duplicates are removed, retain the first record and log the number removed | `customer_id` is defined as a synthetic unique identifier; conflicting duplicates cannot safely be merged without extra information |
| Negative tenure | Remove row | Impossible value under the field definition |
| Negative monthly charge | Remove row | Impossible fictional charge |
| Negative/non-integer support tickets | Remove row | Ticket count cannot be negative or fractional |
| Invalid contract type | Remove row | Must be monthly, quarterly, or annual |
| Invalid churn label | Remove row | Churn must be binary |
| IQR outliers | Flag but retain unless also invalid | An extreme value may be a valid customer observation; deleting it would change the descriptive story |

In [ ]:
# 7. Apply the documented cleaning rules
clean = df.copy()

# 7a. Remove exact duplicate rows.
before = len(clean)
clean = clean.drop_duplicates(keep="first").copy()
removed_exact_duplicates = before - len(clean)

# 7b. Remove rows with missing or impossible required values.
invalid_value_mask = (
    clean[EXPECTED].isna().any(axis=1)
    | (clean["tenure_months"] < 0)
    | (clean["monthly_charge"] < 0)
    | (clean["support_tickets"] < 0)
    | ((clean["support_tickets"] % 1) != 0)
    | (~clean["contract_type"].isin(ALLOWED_CONTRACTS))
    | (~clean["churned"].isin([0, 1]))
)
invalid_value_rows = clean.loc[invalid_value_mask].copy()
clean = clean.loc[~invalid_value_mask].copy()

# 7c. Resolve duplicate customer IDs deterministically because the source defines them as unique.
duplicate_id_mask = clean["customer_id"].duplicated(keep="first")
removed_duplicate_id_rows = clean.loc[duplicate_id_mask].copy()
clean = clean.loc[~duplicate_id_mask].copy()

# 7d. Final analytical dtypes.
clean["tenure_months"] = clean["tenure_months"].astype(int)
clean["support_tickets"] = clean["support_tickets"].astype(int)
clean["monthly_charge"] = clean["monthly_charge"].astype(float)
clean["churned"] = clean["churned"].astype(int)

clean.to_csv(CLEAN_PATH, index=False)

cleaning_log = pd.DataFrame([
    ["Exact duplicate rows", int(removed_exact_duplicates), "Removed duplicate copies; first copy retained."],
    ["Missing/invalid rows", int(len(invalid_value_rows)), "Removed; no unsupported imputation."],
    ["Duplicate customer-ID rows", int(len(removed_duplicate_id_rows)), "Removed after exact duplicates; first ID occurrence retained."],
    ["IQR outliers", int(outlier_report["outlier_count"].sum()), "Flagged for review; retained unless impossible."],
], columns=["issue", "rows_affected", "decision"])

display(cleaning_log)
print("Raw rows:", len(df))
print("Clean rows:", len(clean))

In [ ]:
# 8. Validation checks — all must pass
validation = pd.DataFrame({
    "check": [
        "Required columns present",
        "No missing required values",
        "Customer IDs are unique",
        "No exact duplicate rows",
        "Tenure is non-negative",
        "Monthly charge is non-negative",
        "Support tickets are non-negative integers",
        "Contract type is valid",
        "Churn is binary",
    ],
    "result": [
        set(EXPECTED).issubset(clean.columns),
        int(clean[EXPECTED].isna().sum().sum()) == 0,
        bool(clean["customer_id"].is_unique),
        not bool(clean.duplicated().any()),
        bool((clean["tenure_months"] >= 0).all()),
        bool((clean["monthly_charge"] >= 0).all()),
        bool(((clean["support_tickets"] >= 0) & (clean["support_tickets"] % 1 == 0)).all()),
        bool(clean["contract_type"].isin(ALLOWED_CONTRACTS).all()),
        bool(clean["churned"].isin([0, 1]).all()),
    ]
})

display(validation)
assert validation["result"].all(), "At least one validation check failed. Review the cleaning logic before submission."
print("ALL VALIDATION CHECKS PASSED")

In [ ]:
# 9. Summary statistics and business KPIs
summary_statistics = clean[numeric_cols].describe().T
summary_statistics["median"] = clean[numeric_cols].median()
summary_statistics = summary_statistics[["count", "mean", "std", "min", "25%", "median", "50%", "75%", "max"]]

overall_churn_rate = float(clean["churned"].mean()) if len(clean) else np.nan
overall_retention_rate = 1 - overall_churn_rate if len(clean) else np.nan

kpis = pd.DataFrame({
    "metric": ["Customers", "Churned customers", "Churn rate", "Retention rate", "Average monthly charge", "Total monthly charge among churned customers"],
    "value": [
        len(clean),
        int(clean["churned"].sum()),
        overall_churn_rate,
        overall_retention_rate,
        float(clean["monthly_charge"].mean()),
        float(clean.loc[clean["churned"] == 1, "monthly_charge"].sum()),
    ]
})

display(kpis)
display(summary_statistics)

In [ ]:
# 10. Meaningful distributions
figures = {}

fig, ax = plt.subplots(figsize=(7, 4))
clean["churned"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Customer Churn Distribution")
ax.set_xlabel("Churned (0 = No, 1 = Yes)")
ax.set_ylabel("Customers")
fig.tight_layout()
fig.savefig(DATA_DIR / "01_churn_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

for col, title, filename in [
    ("tenure_months", "Tenure Distribution", "02_tenure_distribution.png"),
    ("monthly_charge", "Monthly Charge Distribution", "03_monthly_charge_distribution.png"),
    ("support_tickets", "Support Tickets Distribution", "04_support_tickets_distribution.png"),
]:
    fig, ax = plt.subplots(figsize=(7, 4))
    clean[col].plot(kind="hist", bins=min(20, max(5, clean[col].nunique())), ax=ax)
    ax.set_title(title)
    ax.set_xlabel(col.replace("_", " ").title())
    ax.set_ylabel("Customers")
    fig.tight_layout()
    fig.savefig(DATA_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

In [ ]:
# 11. Descriptive churn patterns by meaningful customer groups

def churn_table(col):
    out = clean.groupby(col).agg(
        customers=("customer_id", "count"),
        churned=("churned", "sum"),
        churn_rate=("churned", "mean"),
        avg_monthly_charge=("monthly_charge", "mean"),
        avg_tenure_months=("tenure_months", "mean"),
        avg_support_tickets=("support_tickets", "mean"),
    ).reset_index()
    return out.sort_values(["churn_rate", "customers"], ascending=[False, False])

contract_churn = churn_table("contract_type")

tenure_bins = [-np.inf, 6, 12, 24, 48, np.inf]
tenure_labels = ["0–6", "7–12", "13–24", "25–48", "49+"]
clean["tenure_band"] = pd.cut(clean["tenure_months"], bins=tenure_bins, labels=tenure_labels)
tenure_churn = churn_table("tenure_band")

ticket_bins = [-np.inf, 0, 2, 5, np.inf]
ticket_labels = ["0", "1–2", "3–5", "6+"]
clean["support_ticket_band"] = pd.cut(clean["support_tickets"], bins=ticket_bins, labels=ticket_labels)
ticket_churn = churn_table("support_ticket_band")

print("Churn by contract type")
display(contract_churn)
print("Churn by tenure band")
display(tenure_churn)
print("Churn by support-ticket band")
display(ticket_churn)

In [ ]:
# 12. Descriptive revenue-risk-style metric
# This is NOT a forecast and does not imply causation. It is the monthly charge
# associated with customers whose observed churn flag equals 1.
revenue_risk = clean.groupby("churned").agg(
    customers=("customer_id", "count"),
    monthly_charge_sum=("monthly_charge", "sum"),
    avg_monthly_charge=("monthly_charge", "mean"),
).reset_index()
revenue_risk["churned"] = revenue_risk["churned"].map({0: "Retained", 1: "Churned"})
display(revenue_risk)

In [ ]:
# 13. Correlation — descriptive only
corr = clean[["tenure_months", "monthly_charge", "support_tickets", "churned"]].corr(numeric_only=True)
display(corr.round(3))
print("Reminder: correlation is descriptive and does not establish causation.")

In [ ]:
# 14. Build final data-quality summary and export all supporting files
quality_summary = pd.DataFrame([
    ["Source rows", len(df), "Rows in supplied EdVyro CSV after field normalization"],
    ["Cleaned rows", len(clean), "Rows retained after documented rules"],
    ["Exact duplicate rows removed", removed_exact_duplicates, "Exact duplicate copies only"],
    ["Missing/invalid rows removed", len(invalid_value_rows), "No unsupported imputation"],
    ["Duplicate customer-ID rows removed", len(removed_duplicate_id_rows), "First ID occurrence retained because IDs are defined as unique"],
    ["IQR outlier observations flagged", int(outlier_report["outlier_count"].sum()), "Flagged, not automatically deleted"],
    ["Validation checks passed", int(validation["result"].sum()), "All required checks passed"],
], columns=["metric", "value", "decision_or_definition"])

# Save CSV outputs.
profile.to_csv(DATA_DIR / "data_profile.csv", index=False)
cleaning_log.to_csv(DATA_DIR / "cleaning_log.csv", index=False)
quality_summary.to_csv(DATA_DIR / "data_quality_summary.csv", index=False)
outlier_report.to_csv(DATA_DIR / "outlier_audit.csv", index=False)
validation.to_csv(DATA_DIR / "validation_checks.csv", index=False)
summary_statistics.to_csv(DATA_DIR / "summary_statistics.csv")
kpis.to_csv(DATA_DIR / "kpis.csv", index=False)
contract_churn.to_csv(DATA_DIR / "churn_by_contract.csv", index=False)
tenure_churn.to_csv(DATA_DIR / "churn_by_tenure_band.csv", index=False)
ticket_churn.to_csv(DATA_DIR / "churn_by_support_ticket_band.csv", index=False)
revenue_risk.to_csv(DATA_DIR / "observed_monthly_charge_by_churn.csv", index=False)

# Remove temporary analysis-only columns from the final clean CSV.
clean_final = clean[EXPECTED].copy()
clean_final.to_csv(CLEAN_PATH, index=False)

# Excel workbook with the same evidence in one place.
with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    quality_summary.to_excel(writer, sheet_name="Quality_Summary", index=False)
    profile.to_excel(writer, sheet_name="Data_Profile", index=False)
    cleaning_log.to_excel(writer, sheet_name="Cleaning_Log", index=False)
    validation.to_excel(writer, sheet_name="Validation", index=False)
    kpis.to_excel(writer, sheet_name="KPIs", index=False)
    summary_statistics.to_excel(writer, sheet_name="Summary_Statistics")
    outlier_report.to_excel(writer, sheet_name="Outlier_Audit", index=False)
    contract_churn.to_excel(writer, sheet_name="Churn_Contract", index=False)
    tenure_churn.to_excel(writer, sheet_name="Churn_Tenure", index=False)
    ticket_churn.to_excel(writer, sheet_name="Churn_Tickets", index=False)
    revenue_risk.to_excel(writer, sheet_name="Observed_Charge_Risk", index=False)
    clean_final.head(1000).to_excel(writer, sheet_name="Cleaned_Data_Preview", index=False)

print("Created:")
for p in sorted(DATA_DIR.iterdir()):
    print(" -", p.name)

In [ ]:
# 15. Generate a concise submission report using the actual computed results
highest_contract = contract_churn.iloc[0] if len(contract_churn) else None
highest_tenure = tenure_churn.iloc[0] if len(tenure_churn) else None
highest_ticket = ticket_churn.iloc[0] if len(ticket_churn) else None

report = f"""# EdVyro Customer Churn — Data Quality & Exploratory Analysis

## Objective
Create a reproducible, analysis-ready customer churn dataset and a defensible data-quality report.

## Dataset
- Source: supplied EdVyro customer churn CSV
- Raw rows: {len(df):,}
- Cleaned rows: {len(clean):,}
- Fields analyzed: {len(EXPECTED)}

## Quality decisions
- Exact duplicate rows removed: {removed_exact_duplicates:,}
- Missing/invalid rows removed: {len(invalid_value_rows):,}
- Duplicate customer-ID rows removed: {len(removed_duplicate_id_rows):,}
- IQR outlier observations flagged and retained unless impossible: {int(outlier_report['outlier_count'].sum()):,}
- Validation checks passed: {int(validation['result'].sum())}/{len(validation)}

## Key descriptive metrics
- Churn rate: {overall_churn_rate:.1%}
- Retention rate: {overall_retention_rate:.1%}
- Average monthly charge: INR {clean['monthly_charge'].mean():,.2f}
- Monthly charge associated with observed churned customers: INR {clean.loc[clean['churned']==1, 'monthly_charge'].sum():,.2f}

## Initial patterns
- Highest observed churn-rate contract group: {highest_contract['contract_type'] if highest_contract is not None else 'N/A'} ({highest_contract['churn_rate']:.1%})
- Highest observed churn-rate tenure band: {highest_tenure['tenure_band'] if highest_tenure is not None else 'N/A'} ({highest_tenure['churn_rate']:.1%})
- Highest observed churn-rate support-ticket band: {highest_ticket['support_ticket_band'] if highest_ticket is not None else 'N/A'} ({highest_ticket['churn_rate']:.1%})

## Interpretation guardrails
These are descriptive associations in a synthetic dataset. They do not prove that contract type, tenure, or support tickets cause churn, and they are not a predictive model.

## Recommended next measurement
Track churn rate and monthly-charge exposure by the same customer segments in the next observation period to see whether the observed patterns persist.

## Files
The `customer_churn_outputs` folder contains the cleaned CSV, data-quality summary, validation checks, supporting segment tables, charts, and Excel workbook.
"""
REPORT_PATH.write_text(report, encoding="utf-8")
print(report)

## Submission checklist

Before submitting, confirm:

- [ ] Notebook has been run from top to bottom without errors.
- [ ] `ALL VALIDATION CHECKS PASSED` is visible.
- [ ] `customer_churn_cleaned.csv` exists.
- [ ] `data_quality_summary.csv` exists.
- [ ] `customer_churn_data_quality_report.xlsx` exists.
- [ ] Charts are visible.
- [ ] The repository/folder contains this notebook plus `customer_churn_outputs`.
- [ ] The submitted URL is public or view-only.

**Do not invent numbers before running the notebook.** The report is generated from the actual supplied CSV after execution.